# 03. Tree DFS Patterns

Almost every tree problem is solved with **recursion on left and right subtrees**. The skill is deciding **what each call returns**.

**Learning order:** TWO STYLES → DEPTH → INVERT → SYMMETRIC → PATH SUM → BALANCED → DIAMETER → LCA → MAX PATH SUM → INTERVIEW THINKING

## 1. Two styles of tree recursion

- **Top-down:** pass information **down** as parameters (current sum, min/max range, depth).
- **Bottom-up:** each call **returns** information **up** (height, size, a tuple of values). The parent combines the children's answers.

**Global-answer pattern:** sometimes the value a call *returns* is different from the *answer to the problem* (diameter, max path sum). Return one thing to the parent, update a separate `best` variable with the real answer.

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree_from_list(vals):             # level order, None = missing child
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = deque([root]), 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def level_order(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        res.append([])
        for _ in range(len(q)):
            node = q.popleft()
            res[-1].append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return res

def inorder(root):
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

def max_depth(root):
    if not root:
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

def min_depth(root):                              # careful: a node with one child is NOT a leaf
    if not root:
        return 0
    if not root.left:
        return 1 + min_depth(root.right)
    if not root.right:
        return 1 + min_depth(root.left)
    return 1 + min(min_depth(root.left), min_depth(root.right))

print(max_depth(tree_from_list([3, 9, 20, None, None, 15, 7])))                      # 3
print(min_depth(tree_from_list([3, 9, 20, None, None, 15, 7])))                      # 2
print(min_depth(tree_from_list([2, None, 3, None, 4, None, 5, None, 6])))            # 5

3
2
5


## 2. Invert, Same, Symmetric

- **Invert:** swap left and right at every node.
- **Same tree:** both None, or same value and same subtrees.
- **Symmetric:** left subtree is a **mirror** of the right: `a.left` matches `b.right` and `a.right` matches `b.left`.

In [2]:
def invert(root):
    if not root:
        return None
    root.left, root.right = invert(root.right), invert(root.left)
    return root

def same_tree(a, b):
    if not a and not b:
        return True
    if not a or not b or a.val != b.val:
        return False
    return same_tree(a.left, b.left) and same_tree(a.right, b.right)

def is_symmetric(root):
    def mirror(a, b):
        if not a and not b:
            return True
        if not a or not b or a.val != b.val:
            return False
        return mirror(a.left, b.right) and mirror(a.right, b.left)
    return mirror(root.left, root.right) if root else True

print(level_order(invert(tree_from_list([4, 2, 7, 1, 3, 6, 9]))))   # [[4], [7, 2], [9, 6, 3, 1]]
print(same_tree(tree_from_list([1, 2, 3]), tree_from_list([1, 2, 3])))   # True
print(is_symmetric(tree_from_list([1, 2, 2, 3, 4, 4, 3])), is_symmetric(tree_from_list([1, 2, 2, None, 3, None, 3])))   # True False

[[4], [7, 2], [9, 6, 3, 1]]
True
True False


## 3. Path Sum (top-down)

Is there a **root-to-leaf** path with sum `target`? Subtract the node value on the way down. At a **leaf**, check if the remaining target is 0.

In [3]:
def has_path_sum(root, target):
    if not root:
        return False
    if not root.left and not root.right:
        return target == root.val
    return has_path_sum(root.left, target - root.val) or has_path_sum(root.right, target - root.val)

t = tree_from_list([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print(has_path_sum(t, 22), has_path_sum(t, 100))   # True False

True False


## 4. Balanced tree (bottom-up with a sentinel)

Return the **height**, but return `-1` to signal "already unbalanced". This avoids recomputing height at every node (O(n) instead of O(n²)).

In [4]:
def is_balanced(root):
    def height(node):
        if not node:
            return 0
        l = height(node.left)
        if l == -1: return -1
        r = height(node.right)
        if r == -1 or abs(l - r) > 1: return -1
        return 1 + max(l, r)
    return height(root) != -1

print(is_balanced(tree_from_list([3, 9, 20, None, None, 15, 7])))        # True
print(is_balanced(tree_from_list([1, 2, 2, 3, 3, None, None, 4, 4])))    # False

True
False


## 5. Diameter (global-answer pattern)

Diameter = longest path (in edges) between **any two nodes**. At each node, the longest path **through** it = `left height + right height`. But the call returns only `1 + max(left, right)` (the path that the **parent** can extend).

In [5]:
def diameter(root):
    best = 0
    def height(node):
        nonlocal best
        if not node:
            return 0
        l, r = height(node.left), height(node.right)
        best = max(best, l + r)                 # path bending at this node
        return 1 + max(l, r)                    # what the parent can use
    height(root)
    return best

print(diameter(tree_from_list([1, 2, 3, 4, 5])))   # 3

3


## 6. Lowest Common Ancestor (binary tree)

The LCA of `p` and `q` is the deepest node that has both as descendants.

- If `root` is `p` or `q`, return it.
- Ask left and right subtrees.
- If **both** sides return something → `root` is the LCA.
- Otherwise return the non-empty side.

In [6]:
def lca(root, p, q):
    if not root or root is p or root is q:
        return root
    left, right = lca(root.left, p, q), lca(root.right, p, q)
    if left and right:
        return root
    return left or right

def find(root, val):
    if not root: return None
    if root.val == val: return root
    return find(root.left, val) or find(root.right, val)

t = tree_from_list([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
print(lca(t, find(t, 5), find(t, 1)).val, lca(t, find(t, 5), find(t, 4)).val)   # 3 5

3 5


## 7. Binary Tree Maximum Path Sum (hard, same pattern)

A path can start and end at **any** nodes. At each node: best path **bending here** = `val + left gain + right gain`, where a negative gain is ignored (`max(0, ...)`). Return only `val + max(left gain, right gain)` to the parent.

In [7]:
def max_path_sum(root):
    best = float("-inf")
    def gain(node):
        nonlocal best
        if not node:
            return 0
        l, r = max(0, gain(node.left)), max(0, gain(node.right))
        best = max(best, node.val + l + r)
        return node.val + max(l, r)
    gain(root)
    return best

print(max_path_sum(tree_from_list([-10, 9, 20, None, None, 15, 7])), max_path_sum(tree_from_list([1, 2, 3])))   # 42 6

42 6


## 8. Interview thinking

**Ask:** "What does my function **return**? What do I need from the children? Is the answer the returned value, or a global I update?"

**Say:** "Base case is `None`. I combine the left and right results at each node. Time O(n), space O(h)."

**Common mistakes:** leaf vs `None` confusion (min depth, path sum), returning the wrong thing for diameter/max path, recomputing height at every node (O(n²)), not handling negative values (max path sum), comparing nodes by value instead of identity in LCA.

## Quick Revision

- Top-down = parameters. Bottom-up = return values. Global `best` when return value ≠ answer.
- Min depth / path sum: a **leaf** has no children, a node with one child is not a leaf.
- Balanced: return height, `-1` for unbalanced.
- Diameter: `best = max(best, l + r)`, return `1 + max(l, r)`.
- LCA: both sides non-null → root. Max path: ignore negative gains.